In [2]:
import numpy as np
import pandas as pd

import once, this shit is huge - 5g

In [ ]:


train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv', )




/var/folders/5p/tsf09yfn1d7ct362cxy57hyc0000gn/T/ipykernel_19565/2690657249.py:1: DtypeWarning: Columns (0: onpromotion) have mixed types. Specify dtype option on import or set low_memory=False.
  train = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/train.csv')


In [14]:
print(train.shape)
print(train.head())
train['onpromotion'] # onpromotion is boolean but also has NaN values, well need to decide how to clean it.
promotion = train['onpromotion'].copy().drop_duplicates()
promotion

(125497040, 6)
   id        date  store_nbr  item_nbr  unit_sales onpromotion
0   0  2013-01-01         25    103665         7.0         NaN
1   1  2013-01-01         25    105574         1.0         NaN
2   2  2013-01-01         25    105575         2.0         NaN
3   3  2013-01-01         25    108079         1.0         NaN
4   4  2013-01-01         25    108701         1.0         NaN


0             NaN
21657651    False
21668391     True
Name: onpromotion, dtype: object

In [ ]:
total_daily_sales = train.groupby('store_nbr')

In [11]:
item_nbr = train['item_nbr'].copy()
item_nbr.drop_duplicates()


0             103665
1             105574
2             105575
3             108079
4             108701
              ...   
125121287    2122818
125163650    2011459
125163892    2126944
125193855    2123839
125309124    2011451
Name: item_nbr, Length: 4036, dtype: int64

apparently 4036 unique items

notes for holidays:

Wages in the public sector are paid every two weeks on the 15 th and on the last day of the month. Supermarket sales could be affected by this.
A magnitude 7.8 earthquake struck Ecuador on April 16, 2016. People rallied in relief efforts donating water and other first need products which greatly affected supermarket sales for several weeks after the earthquake.

In [31]:
holi = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/holidays_events.csv')
print(holi.shape)
print(holi.head())
clean_holi = holi.drop('description', axis=1)
holi['type'].unique()

(350, 6)
         date     type    locale locale_name                    description  \
0  2012-03-02  Holiday     Local       Manta             Fundacion de Manta   
1  2012-04-01  Holiday  Regional    Cotopaxi  Provincializacion de Cotopaxi   
2  2012-04-12  Holiday     Local      Cuenca            Fundacion de Cuenca   
3  2012-04-14  Holiday     Local    Libertad      Cantonizacion de Libertad   
4  2012-04-21  Holiday     Local    Riobamba      Cantonizacion de Riobamba   

   transferred  
0        False  
1        False  
2        False  
3        False  
4        False  


<StringArray>
['Holiday', 'Transfer', 'Additional', 'Bridge', 'Work Day', 'Event']
Length: 6, dtype: str

In [13]:
items = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/items.csv')
print(items.shape)
print(items.head())

(4100, 4)
   item_nbr        family  class  perishable
0     96995     GROCERY I   1093           0
1     99197     GROCERY I   1067           0
2    103501      CLEANING   3008           0
3    103520     GROCERY I   1028           0
4    103665  BREAD/BAKERY   2712           1


In [ ]:
item_nbr

In [14]:
oil = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/oil.csv')
print(oil.shape)
print(oil.head())

(1218, 2)
         date  dcoilwtico
0  2013-01-01         NaN
1  2013-01-02       93.14
2  2013-01-03       92.97
3  2013-01-04       93.12
4  2013-01-07       93.20


In [23]:
stores = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/stores.csv')
print(stores.shape)
print(stores.head())
clean_stores = stores.drop(['type', 'cluster'], axis=1)
# clean_stores


(54, 5)
   store_nbr           city                           state type  cluster
0          1          Quito                       Pichincha    D       13
1          2          Quito                       Pichincha    D       13
2          3          Quito                       Pichincha    D        8
3          4          Quito                       Pichincha    D        9
4          5  Santo Domingo  Santo Domingo de los Tsachilas    D        4


In [3]:
transactions = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/transactions.csv')
print(transactions.shape)
print(transactions.head())

(83488, 3)
         date  store_nbr  transactions
0  2013-01-01         25           770
1  2013-01-02          1          2111
2  2013-01-02          2          2358
3  2013-01-02          3          3487
4  2013-01-02          4          1922


In [35]:
daily_trans_per_store = transactions.merge(right=clean_stores, on='store_nbr')
# holidays= daily_trans_per_store.copy().merge(right=holi, on=['city', 'state', 'locale_name'])
daily_trans_per_store

daily_trans_total = transactions.groupby('store_nbr')
daily_trans_total

In [4]:
sample_submission = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/sample_submission.csv')
print(sample_submission.shape)
print(sample_submission.head())

(3370464, 2)
          id  unit_sales
0  125497040           0
1  125497041           0
2  125497042           0
3  125497043           0
4  125497044           0


In [5]:
test = pd.read_csv('../data/raw/favorita-grocery-sales-forecasting/test.csv')
print(test.shape)
print(test.head())

(3370464, 5)
          id        date  store_nbr  item_nbr  onpromotion
0  125497040  2017-08-16          1     96995        False
1  125497041  2017-08-16          1     99197        False
2  125497042  2017-08-16          1    103501        False
3  125497043  2017-08-16          1    103520        False
4  125497044  2017-08-16          1    103665        False


some ideas from the data that make sense to me to join: 

item_nbr + family 